# Production d'algues

**Les sources de données de production (volume en tonnes, table pays et table espèces) viennent de la FAO (format CSV)** : https://www.fao.org/fishery/en/collection/global_production?lang=en

In [1]:
import pandas as pd

In [2]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [3]:
df_prod = pd.read_csv('../data/Global_production_quantity.csv')
df_species = pd.read_csv('../data/CL_FI_SPECIES_GROUPS.csv')
df_prod.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1159616 entries, 0 to 1159615
Data columns (total 8 columns):
 #   Column                      Non-Null Count    Dtype  
---  ------                      --------------    -----  
 0   COUNTRY.UN_CODE             1159616 non-null  int64  
 1   SPECIES.ALPHA_3_CODE        1159616 non-null  object 
 2   AREA.CODE                   1159616 non-null  int64  
 3   PRODUCTION_SOURCE_DET.CODE  1159616 non-null  object 
 4   MEASURE                     1159616 non-null  object 
 5   PERIOD                      1159616 non-null  int64  
 6   VALUE                       1159616 non-null  float64
 7   STATUS                      1159616 non-null  object 
dtypes: float64(1), int64(3), object(4)
memory usage: 70.8+ MB


**On ne peut identifier la production qu'uniquement à travers le "species.alpha_3_code"**

In [4]:
df_species.head(10)

,3A_Code,Taxonomic_Code,Identifier,Name_En,Name_Fr,Name_Es,Name_Ar,Name_Cn,Name_Ru,Scientific_Name,...,CPC_Class_Es,CPC_Class_Ar,CPC_Class_Cn,CPC_Class_Ru,CPC_Group_En,CPC_Group_Fr,CPC_Group_Es,CPC_Group_Ar,CPC_Group_Cn,CPC_Group_Ru
0,LAS,1020010XXXXX,2000,Lampreys NEI,Lamproies NCA,Lampreas NEP,NaN,NaN,Миноговые,Petromyzontidae,...,NaN,NaN,NaN,NaN,"Fish live, fresh or chilled for human consumption",NaN,NaN,NaN,NaN,NaN
1,LAR,102001002603,2002,River lamprey,Lamproie de rivière,Lamprea de río,NaN,NaN,Минога речная,Lampetra fluviatilis,...,NaN,NaN,NaN,NaN,"Fish live, fresh or chilled for human consumption",NaN,NaN,NaN,NaN,NaN
2,SBL,103102001401,2003,Bluntnose sixgill shark,Requin griset,Cañabota gris,NaN,NaN,NaN,Hexanchus griseus,...,NaN,NaN,NaN,NaN,"Fish live, fresh or chilled for human consumption",NaN,NaN,NaN,NaN,NaN
3,NTC,103102001801,2004,Broadnose sevengill shark,Platnez,Cañabota gata,NaN,NaN,NaN,Notorynchus cepedianus,...,NaN,NaN,NaN,NaN,"Fish live, fresh or chilled for human consumption",NaN,NaN,NaN,NaN,NaN
4,BSK,106007001001,2005,Basking shark,Pèlerin,Peregrino,NaN,NaN,Акула гигантская,Cetorhinus maximus,...,NaN,NaN,NaN,NaN,"Fish live, fresh or chilled for human consumption",NaN,NaN,NaN,NaN,NaN
5,CCT,106002001001,2006,Sand tiger shark,Requin taureau,Toro bacota,قِرش ثَور,锥齿鲨,NaN,Carcharias taurus,...,NaN,NaN,NaN,NaN,"Fish live, fresh or chilled for human consumption",NaN,NaN,NaN,NaN,NaN
6,THR,1060060010XX,2007,Thresher sharks NEI,Renards de mer NCA,Zorros NEP,NaN,NaN,NaN,Alopias spp,...,NaN,NaN,NaN,NaN,"Fish live, fresh or chilled for human consumption",NaN,NaN,NaN,NaN,NaN
7,ALV,106006001004,2008,Thresher,Renard,Zorro,القرش الثّعلب,狐形长尾鲨,Лисица морская обыкновенная,Alopias vulpinus,...,NaN,NaN,NaN,NaN,"Fish live, fresh or chilled for human consumption",NaN,NaN,NaN,NaN,NaN
8,PTH,106006001001,2009,Pelagic thresher,Renard pélagique,Zorro pelágico,NaN,NaN,NaN,Alopias pelagicus,...,NaN,NaN,NaN,NaN,"Fish live, fresh or chilled for human consumption",NaN,NaN,NaN,NaN,NaN
9,MAK,1060080014XX,2010,Mako sharks,Taupes,Marrajos,السّنفقيات القرشيات,鲭鲨属未定种,NaN,Isurus spp,...,NaN,NaN,NaN,NaN,"Fish live, fresh or chilled for human consumption",NaN,NaN,NaN,NaN,NaN


**Après recherche, c'est dans la colonne "ISSCAAP_group_Fr" qu'on peut repérer les algues et notamment les macro-algues (vertes, brunes, rouges)**

In [5]:
df_species["ISSCAAP_Group_Fr"].value_counts()

ISSCAAP_Group_Fr
Poissons côtiers divers                  3213
Poissons d'eau douce divers              1438
Poissons démersaux divers                1347
Squales, raies, chimères                  888
Ormeaux, bigorneaux, strombes             640
Carpes, barbeaux et autres cyprinidés     616
Clams, coques, arches                     523
Poissons pélagiques divers                425
Crevettes                                 395
Encornets, seiches, poulpes               332
Flets, flétans, soles                     322
Tilapias et autres cichlidés              318
Crabes, araignées de mer                  265
Oursins et autres échinodermes            243
Morues, merlus, églefins                  242
Crustacés d'eau douce                     186
Homards, langoustes                       180
Algues rouges                             180
Coraux                                    166
Harengs, sardines, anchois                147
Crustacés marins divers                   110
Saumons, truites,

On filtre la table d'espèces avec cette information

In [6]:
mask_columns = ["CPC_Class_Es","CPC_Class_Ar","CPC_Class_Cn","CPC_Class_Ru", "ISSCAAP_Group_Cn", "ISSCAAP_Group_Ru", "CPC_Class_Fr","CPC_Group_Fr","CPC_Group_Es", "CPC_Group_Ar","CPC_Group_Cn","CPC_Group_Ru", "ISSCAAP_Group_Es","ISSCAAP_Group_Ar", "Yearbook_Group_Es", "Yearbook_Group_Ar", "Yearbook_Group_Cn", "Yearbook_Group_Ru", "ISSCAAP_Group_En"]
df_species_filter_alguae = df_species.drop(columns=mask_columns)[df_species["ISSCAAP_Group_Fr"].isin(["Algues rouges", "Algues brunes", "Algues vertes"])]
df_species_filter_alguae.head(5)

,3A_Code,Taxonomic_Code,Identifier,Name_En,Name_Fr,Name_Es,Name_Ar,Name_Cn,Name_Ru,Scientific_Name,Author,Major_Group,Yearbook_Group_En,Yearbook_Group_Fr,ISSCAAP_Group_Fr,CPC_Class_En,CPC_Group_En
755,CAU,7451011010XX,2774,Caulerpa seaweeds,Algues caulerpes,Algas caulerpa,ألغيات كوليربا,蕨藻属未定种,NaN,Caulerpa spp,NaN,PLANTAE AQUATICAE,Algae (Aquatic plants),Algues (Plantes aquatiques),Algues vertes,"Seaweeds and other algae, fresh, frozen or dri...",Other aquatic plants and animals
756,UVP,745109101001,2775,Lacy sea lettuce,NaN,NaN,NaN,NaN,NaN,Ulva australis,Areschoug 1854,PLANTAE AQUATICAE,Algae (Aquatic plants),Algues (Plantes aquatiques),Algues vertes,"Seaweeds and other algae, fresh, frozen or dri...",Other aquatic plants and animals
757,LNJ,725115102202,2776,Japanese kelp,Laminaire du Japon,Laminaria del Japón,لمنارية اليابان,海带,NaN,Saccharina japonica,"(Areschoug) C.E.Lane, C.Mayes, Druehl & W.Saun...",PLANTAE AQUATICAE,Algae (Aquatic plants),Algues (Plantes aquatiques),Algues brunes,"Seaweeds and other algae, fresh, frozen or dri...",Other aquatic plants and animals
758,UDP,725103101401,2777,Wakame,Wakamé,Abeto marino,ألغ تنّوبيّ,裙带菜,NaN,Undaria pinnatifida,(Harvey) Suringar 1873,PLANTAE AQUATICAE,Algae (Aquatic plants),Algues (Plantes aquatiques),Algues brunes,"Seaweeds and other algae, fresh, frozen or dri...",Other aquatic plants and animals
759,EOZ,7251161018XX,2778,NaN,NaN,Chascón NEP,NaN,NaN,NaN,Lessonia spp,NaN,PLANTAE AQUATICAE,Algae (Aquatic plants),Algues (Plantes aquatiques),Algues brunes,"Seaweeds and other algae, fresh, frozen or dri...",Other aquatic plants and animals


In [7]:
list_algae_species = df_species_filter_alguae["3A_Code"].tolist()

On a plus qu'à filter le dataframe de production avec cette liste de "3A_code"

In [8]:
df_prod_alguae = df_prod[df_prod["SPECIES.ALPHA_3_CODE"].isin(list_algae_species)]
df_prod_alguae.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9692 entries, 8852 to 1159408
Data columns (total 8 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   COUNTRY.UN_CODE             9692 non-null   int64  
 1   SPECIES.ALPHA_3_CODE        9692 non-null   object 
 2   AREA.CODE                   9692 non-null   int64  
 3   PRODUCTION_SOURCE_DET.CODE  9692 non-null   object 
 4   MEASURE                     9692 non-null   object 
 5   PERIOD                      9692 non-null   int64  
 6   VALUE                       9692 non-null   float64
 7   STATUS                      9692 non-null   object 
dtypes: float64(1), int64(3), object(4)
memory usage: 681.5+ KB


## Traitement des données de pays

In [9]:
df_countries = pd.read_csv('../data/CL_FI_COUNTRY_GROUPS.csv')
df_countries.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 275 entries, 0 to 274
Data columns (total 34 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   UN_Code             275 non-null    int64 
 1   Identifier          275 non-null    int64 
 2   ISO2_Code           257 non-null    object
 3   ISO3_Code           260 non-null    object
 4   Name_En             275 non-null    object
 5   Name_Fr             275 non-null    object
 6   Name_Es             275 non-null    object
 7   Name_Ar             264 non-null    object
 8   Name_Cn             264 non-null    object
 9   Name_Ru             264 non-null    object
 10  Official_Name_En    272 non-null    object
 11  Official_Name_Fr    272 non-null    object
 12  Official_Name_Es    272 non-null    object
 13  Official_Name_Ar    264 non-null    object
 14  Official_Name_Cn    264 non-null    object
 15  Official_Name_Ru    264 non-null    object
 16  Continent_Group_En  263 no

**On décide de "virer" une bonne partie des colonnes notamment celles qui ont des valeurs "null".**

In [10]:
df_countries.columns

Index(['UN_Code', 'Identifier', 'ISO2_Code', 'ISO3_Code', 'Name_En', 'Name_Fr',
       'Name_Es', 'Name_Ar', 'Name_Cn', 'Name_Ru', 'Official_Name_En',
       'Official_Name_Fr', 'Official_Name_Es', 'Official_Name_Ar',
       'Official_Name_Cn', 'Official_Name_Ru', 'Continent_Group_En',
       'Continent_Group_Fr', 'Continent_Group_Es', 'Continent_Group_Ar',
       'Continent_Group_Cn', 'Continent_Group_Ru', 'EcoClass_Group_En',
       'EcoClass_Group_Fr', 'EcoClass_Group_Es', 'EcoClass_Group_Ar',
       'EcoClass_Group_Cn', 'EcoClass_Group_Ru', 'GeoRegion_Group_En',
       'GeoRegion_Group_Fr', 'GeoRegion_Group_Es', 'GeoRegion_Group_Ar',
       'GeoRegion_Group_Cn', 'GeoRegion_Group_Ru'],
      dtype='object')

In [11]:
df_countries_filter = df_countries[["UN_Code", 'Name_Fr', "Continent_Group_Fr"]]
df_countries_filter

,UN_Code,Name_Fr,Continent_Group_Fr
0,51,Arménie,Asie
1,4,Afghanistan,Asie
2,8,Albanie,Europe
3,12,Algérie,Afrique
4,16,Samoa américaines,Océanie
...,...,...,...
270,728,Soudan du Sud,Afrique
271,680,Sercq,Europe
272,729,Soudan,Afrique
273,275,Palestine,Asie


In [12]:
def replace_country_name(string):
  if (string == "République de Corée"):
    return "Corée du Sud"
  else:
    return string

In [13]:
df_countries_filter["Name_Fr"] = df_countries_filter["Name_Fr"].apply(lambda x: replace_country_name(x))

/var/folders/cj/21jsdzgn2fz713n7hrsd75_80000gn/T/ipykernel_69506/2412116511.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_countries_filter["Name_Fr"] = df_countries_filter["Name_Fr"].apply(lambda x: replace_country_name(x))


In [14]:
df_asia = df_countries_filter[df_countries_filter["Continent_Group_Fr"] == "Asie"]
list_asia = df_asia["UN_Code"].tolist()

In [15]:
df_europe = df_countries_filter[df_countries_filter["Continent_Group_Fr"] == "Europe"]
list_europe = df_europe["UN_Code"].tolist()

## Création des graphes algboost

On renomme les types de production pour que ce soit plus simple (juste deux catégories : Récolte et Culture) et quelques autres colonnes.

In [16]:
df_prod_alguae = df_prod_alguae.rename(columns={"PRODUCTION_SOURCE_DET.CODE": "source_production","COUNTRY.UN_CODE": "UN_Code", "PERIOD": "Année","VALUE" : "Production"})
df_prod_alguae["source_production"] = df_prod_alguae["source_production"].apply(lambda x: "Récolte" if x == "CAPTURE" else "Culture")
df_prod_alguae.head(5)

,UN_Code,SPECIES.ALPHA_3_CODE,AREA.CODE,source_production,MEASURE,Année,Production,STATUS
8852,16,SWR,77,Récolte,Q_tlw,2021,0.001,A
8853,16,SWR,77,Récolte,Q_tlw,2022,0.003,A
22896,32,SWB,41,Récolte,Q_tlw,1961,2000.000,A
22897,32,SWB,41,Récolte,Q_tlw,1962,1800.000,A
22898,32,SWB,41,Récolte,Q_tlw,1963,1700.000,A


on fusionne avec le dataframe présentant les pays

In [17]:
df_prod_alguae_country = df_prod_alguae.merge(df_countries_filter, on="UN_Code", how="left")
df_prod_alguae_country.head(5)

,UN_Code,SPECIES.ALPHA_3_CODE,AREA.CODE,source_production,MEASURE,Année,Production,STATUS,Name_Fr,Continent_Group_Fr
0,16,SWR,77,Récolte,Q_tlw,2021,0.001,A,Samoa américaines,Océanie
1,16,SWR,77,Récolte,Q_tlw,2022,0.003,A,Samoa américaines,Océanie
2,32,SWB,41,Récolte,Q_tlw,1961,2000.000,A,Argentine,Amériques
3,32,SWB,41,Récolte,Q_tlw,1962,1800.000,A,Argentine,Amériques
4,32,SWB,41,Récolte,Q_tlw,1963,1700.000,A,Argentine,Amériques


Liste couleurs dispos : ['#1D3B6E','#5B8FCB', '#FDF2ED', '#209490', '#C73175', '#F6A01E', '#1E52A1', '#97C7EC', '#13716A', '#991358', '#E5800B', '#3573B9', '#CDE8FA', '#58B7B0', '#D85F9F', '#F9C06B', '#97C7EC', '#E1D3E9', '#F18882', '#FBDDD9', '#C3ADD4', 'F8C2BB']

 '#0074E4' : FR
 '#1D3B6E' : Chine
 '#209490' : Indonésie
 '#C73175' : Norvège
 '#5B8FCB' : Corée du Sud
 '#C3ADD4' : Philippines
 '#F6A01E' : Autres pays d'Asie
 '#1E52A1' : Autres pays hors Asie
 '#13716A' : Chili
 '#991358' : Inde
 '#E5800B' : Japon
 '#CDE8FA' : Etats-Unis
 '#3573B9' : Autres pays (du monde)
 '#58B7B0' : Irlande
 '#D85F9F' : Islande
 '#F9C06B' : Russie
 '#E1D3E9' : Royaume-Uni
 '#F18882' : Autres pays d'Europe

#### Graphe evolution_production_algues_1950-2022

On doit renommer certains pays et ajouter une catégorie

In [18]:
def get_category(row):
    if row["Continent"] == "Asie":
        if row["Pays_2"] in ["Indonésie", "Philippines", "Corée du Sud", "Chine"]:
            return row["Pays_2"]
        else:
            return "Autres pays d'Asie"
    else:
      return "Autres pays hors Asie"

In [19]:
df_prod_alguae_country_fig_1 = df_prod_alguae_country.rename(columns={"Name_Fr": "Pays_2", "Continent_Group_Fr":"Continent", "Year": "Année"})
df_prod_alguae_country_fig_1.head(5)

,UN_Code,SPECIES.ALPHA_3_CODE,AREA.CODE,source_production,MEASURE,Année,Production,STATUS,Pays_2,Continent
0,16,SWR,77,Récolte,Q_tlw,2021,0.001,A,Samoa américaines,Océanie
1,16,SWR,77,Récolte,Q_tlw,2022,0.003,A,Samoa américaines,Océanie
2,32,SWB,41,Récolte,Q_tlw,1961,2000.000,A,Argentine,Amériques
3,32,SWB,41,Récolte,Q_tlw,1962,1800.000,A,Argentine,Amériques
4,32,SWB,41,Récolte,Q_tlw,1963,1700.000,A,Argentine,Amériques


In [20]:
df_prod_alguae_country_fig_1['Pays'] = df_prod_alguae_country_fig_1.apply(lambda row: get_category(row), axis=1)

In [21]:
# Il s'agit d'une fconction spécifique pour créer les graphes de type "area chart"
def create_multi_traces_area_plot(df,name_color_tuple_list, plot_title):
  fig = go.Figure()
  for name_color_tuple in name_color_tuple_list:
    fig.add_trace(go.Scatter(
      x=df[df["Pays"]== name_color_tuple[0]]["Année"],
      y=df[df["Pays"]== name_color_tuple[0]]["Production"],
      mode='lines',
      line=dict(width=0.5, color=name_color_tuple[1]),
      stackgroup='one',
      fillcolor=name_color_tuple[1],
      name=name_color_tuple[0],
      hovertemplate="""Production : %{y}""" + \
      """<extra></extra>""",
      legendwidth=500,
      hoverlabel=dict(font=dict(family="Parkinsans")),
      )
    )
  fig.update_layout(
    legend_orientation="h",
    hovermode='x unified',
    title_text=plot_title,
    font_family="Parkinsans",
    title_font_weight=800,
    font_weight=600,
    font_color="#1D3B6E",
    paper_bgcolor = 'white',  # Fully transparent background
    plot_bgcolor = 'white',   # Fully transparent plot area
  )
  return fig

In [22]:
df_prod_alguae_country_fig_1_by_year = df_prod_alguae_country_fig_1[["Année", "Pays","Production"]].groupby(["Année", "Pays"]).sum().reset_index()

colors = [('Chine','#1D3B6E'),('Indonésie','#209490'),('Corée du Sud','#5B8FCB'),('Philippines','#C3ADD4'), ("Autres pays d'Asie", '#F6A01E'), ("Autres pays hors Asie",'#1E52A1')]

fig_1 = create_multi_traces_area_plot(df_prod_alguae_country_fig_1_by_year,colors, "Evolution de la production d'algues<br>(en tonnes) par pays")
fig_1

In [27]:
fig_1.write_html("evolution_production_algues_1950-2022_mobile.html", include_plotlyjs="cdn")

#### Graphe "camembert" repartition_culture_recolte Monde et Europe

In [23]:
df_prod_alguae_country_fig_3 = df_prod_alguae_country.rename(columns={"Name_Fr": "Pays", "Continent_Group_Fr":"Continent"})

In [24]:

colors= ['#209490','#5B8FCB' ]
df_prod_alguae_country_fig_3_europe = df_prod_alguae_country_fig_3[df_prod_alguae_country_fig_3["Continent"] == "Europe"]
data_go = df_prod_alguae_country_fig_3[["source_production", "Production"]].groupby("source_production").sum().reset_index()

fig_3 = go.Figure(data=[go.Pie(labels=data_go["source_production"],
                             values=data_go["Production"],
                               pull=[0.2,0],
                               texttemplate = "%{label} <br> %{percent:.0%}",
                               )])
fig_3.update_traces(hoverinfo='skip',textfont_size=14,
                  marker=dict(colors=colors), textposition='outside')
fig_3.update_layout(
    showlegend=False,
    title_text="Répartition entre récolte et<br>culture d'algues dans le monde",
    font_family="Parkinsans",
    title_font_weight=800,
    font_weight=600,
    font_color="#1D3B6E",
)

data_go_bis = df_prod_alguae_country_fig_3_europe[["source_production", "Production"]].groupby("source_production").sum().reset_index()

fig_3_bis = go.Figure(data=[go.Pie(labels=data_go_bis["source_production"],
                             values=data_go_bis["Production"],
                                   pull=[0.2, 0],
                                    texttemplate = "%{label} <br> %{percent:.0%}"
                               )])
fig_3_bis.update_traces(hoverinfo='skip', textfont_size=14,
                  marker=dict(colors=colors), textposition='outside')
fig_3_bis.update_layout(
    font_family="Parkinsans",
    title_font_weight=800,
    font_weight=600,
    font_color="#1D3B6E",
    showlegend=False,
    title_text="Répartition entre récolte et<br>culture d'algues en Europe"
)
display(fig_3,fig_3_bis)
# fig_3
# fig_3 = go.Figure(data=[go.Pie(labels=["Hello", "Bonjour", "Salut"],
#                              values=[1,2,4],pull=[0.2,0]
#                                )])
# fig_3.update_traces(hoverinfo='skip', textinfo='label+percent', textfont_size=14,
#                   marker=dict(colors=colors), textposition='outside')

In [33]:
fig_3.write_html("repartition_culture_recolte_monde_mobile.html", include_plotlyjs="cdn")
fig_3_bis.write_html("repartition_culture_recolte_europe_mobile.html", include_plotlyjs="cdn")

#### Graphe recolte_algues_monde_par_pays

In [25]:
df_prod_alguae_country_fig_4 = df_prod_alguae_country.rename(columns={"Name_Fr": "Pays_2", "Continent_Group_Fr":"Continent"})
df_prod_alguae_country_fig_4_recolte = df_prod_alguae_country_fig_4[df_prod_alguae_country_fig_4["source_production"] == "Récolte"]

In [26]:
def get_category_bis(row):
    if row["Pays_2"] in ["Chili", "Norvège", "Japon", "Inde", "Indonésie", "France"]:
       return row["Pays_2"]
    elif row["Pays_2"].lower() == "états-unis d'amérique":
      return "Etats-Unis"
    else:
      return "Autres pays"

Liste couleurs dispos : ['#1D3B6E','#5B8FCB', '#FDF2ED', '#209490', '#C73175', '#F6A01E', '#1E52A1', '#97C7EC', '#13716A', '#991358', '#E5800B', '#3573B9', '#CDE8FA', '#58B7B0', '#D85F9F', '#F9C06B', '#97C7EC', '#E1D3E9', '#F18882', '#FBDDD9', '#C3ADD4', 'F8C2BB']

 '#0074E4' : FR
 '#1D3B6E' : Chine
 '#209490' : Indonésie
 '#C73175' : Norvège
 '#5B8FCB' : Corée du Sud
 '#97C7EC' : Philippines
 '#F6A01E' : Autres pays d'Asie
 '#1E52A1' : Autres pays hors Asie
 '#13716A' : Chili
 '#991358' : Inde
 '#E5800B' : Japon
 '#CDE8FA' : Etats-Unis
 '#3573B9' : Autres pays (du monde)
 '#58B7B0' : Irlande
 '#D85F9F' : Islande
 '#F9C06B' : Russie
 '#E1D3E9' : Royaume-Uni
 '#F18882' : Autres pays d'Europe

In [27]:
df_prod_alguae_country_fig_4_recolte['Pays'] = df_prod_alguae_country_fig_4_recolte.apply(lambda row: get_category_bis(row), axis=1)
df_prod_alguae_country_fig_4_recolte_by_year = df_prod_alguae_country_fig_4_recolte[["Année", "Pays","Production"]].groupby(["Année", "Pays"]).sum().reset_index()
colors = [('Chili','#13716A'),('Norvège','#C73175'),("Indonésie",'#209490'),('Inde','#991358'), ('Japon', '#E5800B'), ('France','#0074E4'), ("Etats-Unis", '#CDE8FA'), ('Autres pays', '#3573B9')]

fig_4 = create_multi_traces_area_plot(df_prod_alguae_country_fig_4_recolte_by_year,colors, "Récolte d'algues dans le monde<br>(en tonnes) par pays")
fig_4

/var/folders/cj/21jsdzgn2fz713n7hrsd75_80000gn/T/ipykernel_69506/2723511226.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_prod_alguae_country_fig_4_recolte['Pays'] = df_prod_alguae_country_fig_4_recolte.apply(lambda row: get_category_bis(row), axis=1)


In [41]:
fig_4.write_html("recolte_algues_monde_par_pays_mobile.html", include_plotlyjs="cdn")

#### Graphe production_algues_europe_par_pays

In [28]:
def get_country_europe(row):
    if row["Pays_2"] in ["Norvège", "France", "Irlande", "Islande"]:
       return row["Pays_2"]
    elif row["Pays_2"] == "Fédération de Russie":
      return "Russie"
    elif row["Pays_2"] == "Royaume-Uni de Grande-Bretagne et d'Irlande du Nord":
      return "Royaume-Uni"
    else:
      return "Autres pays d'Europe"

In [29]:
df_prod_alguae_europe = df_prod_alguae[df_prod_alguae["UN_Code"].isin(list_europe)].rename(columns={"COUNTRY.UN_CODE": "UN_Code", "PERIOD": "Année","VALUE" : "Production"}).merge(df_countries_filter, on="UN_Code", how="left").rename(columns={"Name_Fr": "Pays_2", "Continent_Group_Fr": "Continent"})
df_prod_alguae_europe["Pays"] = df_prod_alguae_europe.apply(lambda row: get_country_europe(row), axis=1)
df_prod_alguae_europe_country_by_year = df_prod_alguae_europe[["Année", "Pays", "Production"]].groupby(["Année", "Pays"]).sum().reset_index()

colors = [('Norvège','#C73175'),('France','#0074E4'),('Irlande','#58B7B0'),('Islande','#D85F9F'), ("Russie",'#F9C06B'), ('Royaume-Uni','#E1D3E9'), ("Autres pays d'Europe",'#F18882')]

fig_5 = create_multi_traces_area_plot(df_prod_alguae_europe_country_by_year,colors, plot_title="Production d'algues en Europe<br>(en tonnes) par pays")
fig_5

In [48]:
fig_5.write_html("production_algues_europe_par_pays_mobile.html", include_plotlyjs="cdn")

#### Graphe especes_algues_produites_europe

**Les données ici sont récupérées de ce Google sheet : https://docs.google.com/spreadsheets/d/1bhuzeFcvNo5S55Zbi_yOA3IuDdaUHtkU_QoQnPsOTC0/edit?gid=2037908735#gid=2037908735**

**La source originelle est ici : https://data.jrc.ec.europa.eu/collection/id-00363**

On crée le dataframe pour les espèces produites en Europe en culture

In [ ]:
list_algae_culture = ['Alaria sp.','Ascophyllum nodosum','Asparagopsis sp.','Brown seaweed','Calliblepharis jubata','Caulerpa sp.','Chondrus sp.','Codium sp.','Falkenbergia sp.','Fucus sp.','Gracilaria sp.','Gracilariopsis longissima','Himanthalia sp.','Kappaphycus','Laminaria sp.','Palmaria sp.','Porphyra sp.','Saccharina sp.','Schizymenia jonssonii','Ulva sp.','Ulvella lens','Undaria sp.','Espèce inconnue','Vertebrata lanosa']
list_nb_culture = [17,1,1,2,1,1,2,2,1,5,3,2,2,1,10,8,2,29,1,15,1,3,3,1]

In [54]:
df_culture = pd.DataFrame({"algae":list_algae_culture, "nb_culture": list_nb_culture })
df_culture_sort = df_culture.sort_values(by="nb_culture", ascending=False)
df_culture_sort

,algae,nb_culture
17,Saccharina sp.,29
0,Alaria sp.,17
19,Ulva sp.,15
14,Laminaria sp.,10
15,Palmaria sp.,8
9,Fucus sp.,5
22,Espèce inconnue,3
21,Undaria sp.,3
10,Gracilaria sp.,3
16,Porphyra sp.,2


On crée le dataframe pour les espèces produites en Europe en récolte

In [55]:
list_algae_recolte = ['Alaria sp.','Ascophyllum nodosum','Asparagopsis sp.','Bifurcaria bifurcata','Brown seaweed','Calcareous algae','Chondrus sp.','Chorda filum','Coccotylus truncatus','Codium sp.','Corallina sp.','Cystoseira sp.','Delesseria sanguinea','Dilsea carnosa','Fucus sp.','Furcellaria lumbricalis','Gelidium sp.','Gigartina sp.','Gracilaria sp.','Grateloupia turuturu','Green seaweed','Halopteris scoparia','Himanthalia sp.','Laminaria sp.','Lithothamnium calcareum','Mastocarpus stellatus','Osmundea pinnatifida','Padina pavonica','Palmaria sp.','Pelvetia sp.','Petalonia binghamiae','Porphyra sp.','Pterocladiella capillacea','Red seaweed','Saccharina sp.','Salicornia sp.','Sargassum sp.','Solieria sp.','Ulva sp.','Undaria sp.','Espèce inconnue','Vertebrata lanosa','Zonaria tournefortii']
list_nb_recolte =[13,25,2,1,7,1,21,1,1,6,1,1,1,1,30,3,3,3,1,1,2,1,30,37,4,3,5,1,36,1,1,25,1,3,26,2,2,1,32,22,10,4,1]

In [56]:
df_autre_recolte = pd.DataFrame({"algae":list_algae_recolte, "nb_recolte": list_nb_recolte })
df_autre_recolte

,algae,nb_recolte
0,Alaria sp.,13
1,Ascophyllum nodosum,25
2,Asparagopsis sp.,2
3,Bifurcaria bifurcata,1
4,Brown seaweed,7
5,Calcareous algae,1
6,Chondrus sp.,21
7,Chorda filum,1
8,Coccotylus truncatus,1
9,Codium sp.,6


In [59]:
df_autre_recolte_sort = df_autre_recolte.sort_values(by="nb_recolte", ascending=False)
df_autre_recolte_sort

,algae,nb_recolte
23,Laminaria sp.,37
28,Palmaria sp.,36
38,Ulva sp.,32
14,Fucus sp.,30
22,Himanthalia sp.,30
34,Saccharina sp.,26
31,Porphyra sp.,25
1,Ascophyllum nodosum,25
39,Undaria sp.,22
6,Chondrus sp.,21


In [60]:
colors_recolte = ['#1D3B6E','#5B8FCB', '#FDF2ED', '#209490', '#C73175', '#F6A01E', '#1E52A1', '#97C7EC', '#13716A', '#991358', '#E5800B', '#3573B9', '#CDE8FA', '#58B7B0', '#D85F9F', '#F9C06B', '#97C7EC', '#E1D3E9', '#F18882', '#FBDDD9', '#C3ADD4', 'F8C2BB']
colors_culture = ['#F6A01E','#E5800B', '#FDF2ED', '#1D3B6E', '#5B8FCB','#209490' , '#3573B9', '#13716A','#F9C06B', '#1E52A1','#D85F9F', '#C73175', '#58B7B0', '#991358', '#CDE8FA','#97C7EC'  , '#97C7EC', '#E1D3E9', '#F18882', '#FBDDD9', '#C3ADD4', 'F8C2BB']

In [68]:

## Récolte
fig_6 = go.Figure()
fig_6.add_trace(go.Treemap(
    labels = df_autre_recolte_sort["algae"],
    parents = ["",] *15,
    values =  df_autre_recolte_sort["nb_recolte"],
    texttemplate="""<span style="font-weight:800">%{label}</span> <br>""" + \
    """%{value} entreprises (%{percentEntry:.1%})""",
    marker_colors= colors_recolte,
    root_color="white"

))
fig_6.update_layout(
    title_text="Espèces d'algues produites<br>en Europe (Récolte)",
    font_family="Parkinsans",
    title_font_weight=800,
    font_weight=600,
    font_color="#1D3B6E",
  )
fig_6.update_traces(marker=dict(cornerradius=5), hoverinfo="skip")

## Culture
fig_6_bis = go.Figure()
fig_6_bis.add_trace(go.Treemap(
    labels = df_culture_sort["algae"],
    parents = ["",] *15,
    # parents = ["",] *24,
    values = df_culture_sort["nb_culture"],
     texttemplate="""<span style="font-weight:800">%{label}</span> <br>""" + \
    """%{value} entreprises (%{percentEntry:.1%})""",
    marker_colors= colors_culture,
    root_color="white",

))
fig_6_bis.update_layout(
    title_text="Espèces d'algues produites<br>en Europe (Culture)",
    font_family="Parkinsans",
    title_font_weight=800,
    font_weight=600,
    font_color="#1D3B6E",
  )

fig_6_bis.update_traces(marker=dict(cornerradius=5), hoverinfo="skip")
display(
    fig_6,
    fig_6_bis
)

In [69]:
fig_6.write_html("especes_algues_produites_europe_recolte_mobile.html", include_plotlyjs="cdn")

In [70]:
fig_6_bis.write_html("especes_algues_produites_europe_culture_mobile.html", include_plotlyjs="cdn")

#### Graphe projection_emplois_2030

**La source des données est issue de ce rapport : https://phyconomy.net/wp-content/uploads/2020/10/Seaweed_for_Europe-Hidden_Champion_of_the_ocean-Report.pdf**

In [ ]:

x=['Emplois<br>directs', 'Emplois<br>indirects', 'Emplois<br>induits', 'Total<br>Emplois']
df =pd.DataFrame({"x":x, "y":[0, 37, 65, 0], "z":[22,17,11,50], "a": [15, 11, 7, 33] })
# df

fig_7 = go.Figure(go.Bar(x=df["x"], y=df["y"], name='',marker_color='white')
    )
fig_7.add_trace(go.Bar(x=df["x"], y=df["z"], name='Produits cultivés et finis en Europe', marker_color='#F6A01E', text=df["z"], textfont=dict(color="white",size=10)) )
fig_7.add_trace(go.Bar(x=df["x"], y=df["a"], name='Produits cultivés hors Europe, finis en Europe',marker_color='#209490', text=df["a"], textfont=dict(color="white", size=10)))

fig_7.update_layout(barmode='stack',
                  legend_orientation='h',
                  paper_bgcolor = 'white',  # Fully transparent background
                  plot_bgcolor = 'white',   # Fully transparent plot area
                  title_text="Emplois potentiels créés<br>(milliers de ETP) en Europe<br>par l'industrie des algues en 2030",
                  font_family="Parkinsans",
                  title_font_weight=800,
                  font_weight=600,
                  font_color="#1D3B6E",
    )

fig_7.update_yaxes(showline=False, visible=False)
fig_7.update_xaxes(showline=True, linewidth=2, linecolor='#1D3B6E', tickfont=dict(size=10))
fig_7.update_traces(hoverinfo="skip", marker=dict(
                              line=dict(width=1,
                                        color='rgba(0,0,0,0)'),
                              ), textposition='inside'
)
fig_7.show()

In [77]:
fig_7.write_html("projection_emplois_2030_mobile.html", include_plotlyjs="cdn")

#### Graphe chiffre_affaires_europe_par_pays

**Les données ici sont issues du google Sheet suivant : https://docs.google.com/spreadsheets/d/15DgqS1Lia5SoO2-rK6bCOh41MroGS3ZJo90b-S0PWXo/edit?pli=1&gid=1772352892#gid=1772352892**

**La source originelle est ici : https://data.jrc.ec.europa.eu/collection/id-00363**

In [ ]:
df_ca_alguae = pd.read_csv('../data/Algae-industry-Europe-socioeconomic - Socio-economic data.csv')
df_ca_alguae.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 548 entries, 0 to 547
Data columns (total 11 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   ID                                              548 non-null    object 
 1   Country                                         548 non-null    object 
 2   Organism group                                  548 non-null    object 
 3   Step in value chain                             548 non-null    object 
 4   Average available turnover in the last 5 years  253 non-null    object 
 5   Number of employees                             312 non-null    object 
 6   % of business focused on algae - ESTIMATED      380 non-null    object 
 7   % of business focused on algae - REFERENCE      317 non-null    object 
 8   Is Algae considered the main business stream    380 non-null    object 
 9   Unnamed: 9                                 

In [80]:
df_ca_alguae.rename(columns={"Organism group": "production_type", "Average available turnover in the last 5 years": "turnover_l5y"}).head(10)

,ID,Country,production_type,Step in value chain,turnover_l5y,Number of employees,% of business focused on algae - ESTIMATED,% of business focused on algae - REFERENCE,Is Algae considered the main business stream,Unnamed: 9,Unnamed: 10
0,AT01,Austria,Microalgae,Producing & processing,"16,061",15,100%,Website,Yes,NaN,NaN
1,AT02,Austria,Microalgae & Spirulina,Producing & processing,NaN,15,100%,Website,Yes,NaN,NaN
2,AT03,Austria,Spirulina,Producing & processing & services,NaN,5,100%,Website,Yes,NaN,NaN
3,AT04,Austria,Microalgae,Services,"6,350,815,930","27,232",2%,Website,No,NaN,NaN
4,AT05,Austria,Macroalgae,Processing,"6,754,249",37,33%,Website,No,NaN,NaN
5,AT06,Austria,Microalgae,Services,"88,460,310",292,2%,Website,No,NaN,NaN
6,BE01,Belgium,Microalgae,Producing & processing & services,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,BE02,Belgium,Microalgae,Producing,"552,097",5,100%,Website,Yes,NaN,NaN
8,BE03,Belgium,Spirulina,Producing & processing,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,BE04,Belgium,Algae sensu lato,Processing & services,"16,486,171",101,5%,Website,No,NaN,NaN


On renomme certaines colonnes

In [81]:
df_ca_alguae_bis = df_ca_alguae.rename(columns={"Organism group": "production_type", "Average available turnover in the last 5 years": "turnover_l5y", "Is Algae considered the main business stream": "is_algae_business", "% of business focused on algae - ESTIMATED": "algae_business_percent_estimated"})

On filtre les lignes en visant ls entreprises centrés sur les macro-algues uniqument et si leur business des algues est bien leur business principale

In [83]:
df_ca_alguae_macro_clean = df_ca_alguae_bis[["ID", "Country", "production_type", "turnover_l5y", "is_algae_business"]]
df_ca_alguae_macro_clean.head(10)

,ID,Country,production_type,turnover_l5y,is_algae_business
0,AT01,Austria,Microalgae,"16,061",Yes
1,AT02,Austria,Microalgae & Spirulina,NaN,Yes
2,AT03,Austria,Spirulina,NaN,Yes
3,AT04,Austria,Microalgae,"6,350,815,930",No
4,AT05,Austria,Macroalgae,"6,754,249",No
5,AT06,Austria,Microalgae,"88,460,310",No
6,BE01,Belgium,Microalgae,NaN,NaN
7,BE02,Belgium,Microalgae,"552,097",Yes
8,BE03,Belgium,Spirulina,NaN,NaN
9,BE04,Belgium,Algae sensu lato,"16,486,171",No


In [85]:
df_ca_alguae_macro_clean_by_country = df_ca_alguae_macro_clean[(df_ca_alguae_macro_clean['is_algae_business'] == "Yes")&(df_ca_alguae_macro_clean['production_type'] == "Macroalgae")]
df_ca_alguae_macro_clean_by_country["turnover_l5y"] = df_ca_alguae_macro_clean_by_country["turnover_l5y"].str.replace(',', "").fillna(0)
df_ca_alguae_macro_clean_by_country["turnover_l5y"] = df_ca_alguae_macro_clean_by_country["turnover_l5y"].astype('int64')
df_ca_alguae_macro_clean_by_country = df_ca_alguae_macro_clean_by_country[["Country", "turnover_l5y"]].rename(columns={"turnover_l5y": "Chiffre_d_affaires", "Country": "Pays"}).groupby(by="Pays").sum().reset_index().sort_values(by="Chiffre_d_affaires", ascending=False)
df_ca_alguae_macro_clean_by_country

/tmp/ipykernel_16607/2087684375.py:2: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/tmp/ipykernel_16607/2087684375.py:3: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



,Pays,Chiffre_d_affaires
2,France,54826349
4,Ireland,40403176
5,Norway,16196891
3,Iceland,8278434
7,Spain,7030840
6,Portugal,1045143
1,Estonia,855025
11,UK,800383
0,Denmark,23984
8,Sweden,23931


In [86]:
df_ca_alguae_macro_by_country = df_ca_alguae_macro_clean_by_country[df_ca_alguae_macro_clean_by_country["Chiffre_d_affaires"] > 0]

 '#0074E4' : FR
 '#1D3B6E' : Chine
 '#209490' : Indonésie
 '#C73175' : Norvège
 '#5B8FCB' : Corée du Sud
 '#C3ADD4' : Philippines
 '#F6A01E' : Autres pays d'Asie
 '#1E52A1' : Autres pays hors Asie
 '#13716A' : Chili
 '#991358' : Inde
 '#E5800B' : Japon
 '#CDE8FA' : Etats-Unis
 '#3573B9' : Autres pays (du monde)
 '#58B7B0' : Irlande
 '#D85F9F' : Islande
 '#F9C06B' : Russie
 '#E1D3E9' : Royaume-Uni
 '#F18882' : Autres pays d'Europe

In [87]:
def translate_country(row):
  if (row["Pays"] == "Ireland"):
    return 'Irlande'
  elif (row["Pays"] == "Norway"):
    return 'Norvège'
  elif (row["Pays"] == "Iceland"):
    return 'Islande'
  elif (row["Pays"] == "Spain"):
    return 'Espagne'
  elif (row["Pays"] == "Estonia"):
    return 'Estonie'
  elif (row["Pays"] == "UK"):
    return 'Royaume-Uni'
  elif (row["Pays"] == "Denmark"):
    return 'Danemark'
  elif (row["Pays"] == "Sweden"):
    return 'Suède'
  else:
    return row["Pays"]

In [91]:
colors= ['#0074E4','#58B7B0', '#C73175', '#D85F9F', '#991358','#F6A01E', '#1E52A1', '#E1D3E9', '#13716A','#E5800B', '#3573B9', '#CDE8FA', '#58B7B0', '#D85F9F', '#F9C06B', '#97C7EC', '#E1D3E9', '#F18882', '#FBDDD9', '#C3ADD4', '#F8C2BB']
df_ca_alguae_macro_by_country["Pays"] = df_ca_alguae_macro_by_country.apply(translate_country, axis=1)
fig_10 = go.Figure(
    data=[
        go.Bar(x=df_ca_alguae_macro_by_country["Pays"],
               y=df_ca_alguae_macro_by_country["Chiffre_d_affaires"],
               marker_color=colors,
               hovertemplate="""<span style='font-family:Parkinsans; font-weight:600'>Pays : %{x}</span><br>""" + \
               """<span style='font-family:Parkinsans; font-weight:600'>Chiffre d'affaires : %{y}</span>""" + \
      """<extra></extra>""",
        )
    ],
    layout=dict(
        barcornerradius=15,
    ),
)
fig_10.update_layout(
    title_text="Chiffre d’affaires réalisé (en euros)<br>par pays en Europe grâce à la<br>production d’algues",
    font_family="Parkinsans",
    title_font_weight=800,
    font_weight=600,
    font_color="#1D3B6E",
    paper_bgcolor = 'white',  # Fully transparent background
    plot_bgcolor = 'white',   # Fully transparent plot area

  )
fig_10.show()

/tmp/ipykernel_16607/1719756322.py:2: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [92]:
fig_10.write_html("chiffre_affaires_europe_par_pays_mobile.html", include_plotlyjs="cdn")